In [1]:
import pandas as pd
import numpy as np
import os

import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow import keras

### Neural networks can find groups or pattern and pull out meanings.

# How do we choose layers, neurons and hyperparamaters?
## 1) Use training performance to guide your decisions
#### => High accuracy on training but not validation (Overfil) : Reduce # of paramaters
#### => Low accuracy on training :  Reduce # of paramaters

## 2) Automatically search for the best hyperparamaters with grid search (learning rate, batch size, optimizer, dropout, etc.) 

# Why do we need an activation function?

## 1) To introduce non-linearity into our neural net calculations
## 2) This is what allows us to fit more complex data & compute some really exciting things.

###### Relu helps avoid vanishing gradient problem 

1. Why use the "Information Funnel" (128 → 64 → 32)?
Think of a neural network like a detective agency.

The First Layer (128 neurons): This layer looks at the raw data. It needs a lot of "eyes" (neurons) to catch every tiny detail, edge, and shadow.

The Middle Layer (64 neurons): It takes the details from the first layer and starts combining them. It doesn't need to see the raw pixels anymore; it only needs to see the "concepts" (like "this is a circle" or "this is a line").

The Final Hidden Layer (32 neurons): It condenses those concepts into high-level features (like "this is a face").
The Goal: You are forcing the model to compress the information. If you kept 128 neurons in every layer, the model might just "memorize" the noise in the data. By squeezing the data through a smaller "neck," you force it to keep only the most important patterns.


2. Other "Unspoken Rules" for LayersRule of Powers of TwoYou’ll notice people almost always use 32, 64, 128, 256, or 512.The "Why": This isn't for the AI's "brain"—it's for your Computer's hardware. Modern GPUs and CPUs are optimized to process data in chunks of $2^n$. Using these numbers makes your training run significantly faster because the memory alignment is perfect.The "Bottleneck" RuleNever make a hidden layer smaller than your output layer.

The "Why": If you are trying to classify 10 different types of clothes, but you have a hidden layer with only 3 neurons, you have created a "bottleneck." You are throwing away too much information before the model can make a final decision.

3. Start with "ReLu", End with "Softmax" or "Sigmoid"Hidden Layers: Use ReLU (Rectified Linear Unit). It’s fast and prevents the "vanishing gradient" problem where the model stops learning.

Output Layer: 
* Use Sigmoid if it's a Yes/No question (Binary).Use Softmax if you are choosing between multiple categories (like Cat vs Dog vs Bird), because it makes the total probability add up to 100%.Don't Go Too Deep Too FastFor most beginner projects (like recognizing handwriting or basic CSV data), two to three hidden layers are plenty. Adding 10 layers sounds like it would make the model "smarter," but it actually makes it much harder to train and prone to "overfitting."



In [2]:
os.getcwd()

'C:\\Users\\newta\\OneDrive\\Desktop\\REPO\\Python-Frameworks-Tutorials\\INTRO_TO_NEURAL_NETWORKS'

In [3]:
os.listdir('.')

['.ipynb_checkpoints',
 'clusters',
 'clusters_two_categories',
 'complex',
 'Images',
 'linear',
 'Neural Net.ipynb',
 'quadratic']

In [4]:
os.listdir('./linear')

['data', 'figure.png', 'network_linear.py']

In [5]:
os.listdir('./linear/data')

['test.csv', 'train.csv']

# Linear

In [6]:
train_df = pd.read_csv('./linear/data/train.csv')

In [7]:
train_df.head()

,x,y,color
0,1.146728,2.233629,0.0
1,3.676886,4.520687,0.0
2,0.730671,1.426260,0.0
3,1.950790,3.145987,0.0
4,4.323010,5.320534,0.0


In [8]:
# It is important to shuffle the data, not like att 0 color at the topm, and all 1 at the bottom.
np.random.shuffle(train_df.values)

In [9]:
train_df.head(15)

,x,y,color
0,0.281411,1.672005,0.0
1,0.236442,1.154233,0.0
2,2.792829,1.713031,1.0
3,4.914150,5.581220,0.0
4,2.260000,3.438216,0.0
5,2.396349,1.577377,1.0
6,0.107700,1.207857,0.0
7,4.830714,5.711982,0.0
8,2.606658,3.207704,0.0
9,0.195739,0.532447,0.0


In [10]:
model = keras.Sequential([
    keras.layers.Dense(4,input_shape=(2,), activation = 'relu'),
    keras.layers.Dense(2)
])

C:\Users\newta\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1. Correcting the MathYou are exactly right about the error. Let's look at your examples:Scenario A: from_logits=TrueThe loss function takes your raw numbers (5.2, -1.3), internally runs the Softmax formula to turn them into probabilities (0.998, 0.002), and then does the math:$$Loss = -(0 \cdot \log(0.998) + 1 \cdot \log(0.002))$$This works perfectly because $\log$ of a positive decimal is a valid number.Scenario B: from_logits=False (The Error)The loss function assumes the numbers you sent are already probabilities. It tries to calculate:$$Loss = -(0 \cdot \log(5.2) + 1 \cdot \log(-1.3))$$The Crash: You cannot take the logarithm of a negative number ($-1.3$). This is why your model would return NaN (Not a Number) or throw an error.2. Does it automatically apply Softmax?Yes and No—and this is a very important distinction:During Training: Yes. When calculating the Loss, Keras automatically applies Softmax internally to figure out the error and update the weights.During Prediction (model.predict): No. This is the "catch." If you use from_logits=True and then later try to use your model to predict a new value, model.predict(new_data) will still give you raw numbers like [5.2, -1.3]. It won't look like probabilities unless you manually wrap the output in a Softmax function or add the layer back in later.3. Summary of the WorkflowWhen you use your current model with from_logits=True:Last Layer: Spits out raw "Logits" (e.g., 10.5, -2.3).Loss Function: Sees from_logits=True, converts those to probabilities (e.g., 0.99, 0.01), and calculates the error.Optimizer: Uses that error to fix the weights.

In [11]:
model.compile(optimizer = 'adam', 
              loss = keras.losses.SparseCategoricalCrossentropy(from_logits = True),
             metrics = ['accuracy'])

In [12]:
print(type(train_df.x.values))

<class 'numpy.ndarray'>


In [13]:
x = np.column_stack((train_df.x.values, train_df.y.values))

In [14]:
type(x)

numpy.ndarray

In [15]:
x[0:5]

array([[0.2814112 , 1.67200462],
       [0.23644247, 1.15423278],
       [2.79282922, 1.71303066],
       [4.9141499 , 5.58122001],
       [2.25999959, 3.43821644]])

In [16]:
model.fit(x, train_df.color.values, batch_size = 4)

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8890 - loss: 0.3146


### Evaluating the test data

In [17]:
test_df = pd.read_csv('./linear/data/test.csv')

In [18]:
# np.random.shuffle(test_df.color.values)

In [19]:
test_df.head()

,x,y,color
0,2.684292,3.867108,0.0
1,2.707883,4.002614,0.0
2,2.705905,3.859686,0.0
3,4.536191,5.240051,0.0
4,3.656068,4.461771,0.0


In [20]:
test_x = np.column_stack((test_df.x.values, test_df.y.values))

In [21]:
model.evaluate(test_x, test_df.color.values)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0685


[0.0685119703412056, 1.0]